In [2]:
import pandas as pd
import numpy as np

In [3]:

save_path = 'cleaned dataset/'

api_data_aadhar_enrolment = [
    "api_data_aadhar_enrolment_0_500000.csv",
    "api_data_aadhar_enrolment_500000_1000000.csv",
    "api_data_aadhar_enrolment_1000000_1006029.csv"
]

api_data_aadhar_demographic = [
    "api_data_aadhar_demographic_0_500000.csv",
    "api_data_aadhar_demographic_500000_1000000.csv",
    "api_data_aadhar_demographic_1000000_1500000.csv",
    "api_data_aadhar_demographic_1500000_2000000.csv",
    "api_data_aadhar_demographic_2000000_2071700.csv"
]

api_data_aadhar_biometric = [
    "api_data_aadhar_biometric_0_500000.csv",
    "api_data_aadhar_biometric_500000_1000000.csv",
    "api_data_aadhar_biometric_1000000_1500000.csv",
    "api_data_aadhar_biometric_1500000_1861108.csv"
]

aadhar_dataset = [ 
    'aadhar dataset/api_data_aadhar_biometric/' ,
    'aadhar dataset/api_data_aadhar_demographic/',
    'aadhar dataset/api_data_aadhar_enrolment/'
]

In [4]:
rename_map = {
        'age_0_5': 'age between 0 and 5',
        'age_5_17': 'age between 5 and 17',
        'age_18_greater': 'age 17 and above',
        'demo_age_5_17': 'age between 5 and 17',
        'demo_age_17_': 'age 17 and above',
        'bio_age_5_17': 'age between 5 and 17',
        'bio_age_17_': 'age 17 and above'
    }

state_map = {
        # Renamed States
        'orissa': 'odisha',
        'pondicherry': 'puducherry',
        'uttaranchal': 'uttarakhand',
        'chhatisgarh': 'chhattisgarh',
        
        # Typos 
        'west  bengal': 'west bengal',
        'west bangal': 'west bengal',
        'west bengli': 'west bengal',
        'westbengal': 'west bengal',
        'uttaranchal': 'uttarakhand',
        'chhatisgarh': 'chhattisgarh',
        'tamilnadu': 'tamil nadu',
        
        # Merged Union Territories (Dadra & Nagar Haveli + Daman & Diu)
        'dadra and nagar haveli': 'dadra and nagar haveli and daman and diu',
        'daman and diu': 'dadra and nagar haveli and daman and diu',
        'the dadra and nagar haveli and daman and diu':'dadra and nagar haveli and daman and diu',

        
        # Districts/Cities accidentally listed as States
        'darbhanga': 'bihar',
        'jaipur': 'rajasthan',
        'nagpur': 'maharashtra',
        'balanagar': 'telangana',
        'madanapalle': 'andhra pradesh',
        'puttenahalli': 'karnataka',
        'raja annamalai puram': 'tamil nadu',
        
        # Previous Fixes (Just to be safe)
        'tamilnadu': 'tamil nadu',
        'orissa': 'odisha',
        'pondicherry': 'puducherry',
        'uttaranchal': 'uttarakhand',
        'chhatisgarh': 'chhattisgarh',

    }

In [5]:
state_prefixes = {
    'delhi': ['11'],
    'haryana': ['12', '13'],
    'punjab': ['14', '15', '16'],
    'chandigarh': ['16'],
    'himachal pradesh': ['17'], 
    'jammu and kashmir': ['18', '19'], 
    'ladakh': ['19'],
    'uttar pradesh': [str(i) for i in range(20, 29)], 
    'uttarakhand': ['24', '25', '26'], 
    'rajasthan': [str(i) for i in range(30, 35)], 
    'gujarat': [str(i) for i in range(36, 40)],
    'daman and diu': ['39'], 
    'dadra and nagar haveli': ['39'],
    'dadra and nagar haveli and daman and diu': ['39'],
    'maharashtra': [str(i) for i in range(40, 45)], 
    'goa': ['40'],
    'madhya pradesh': [str(i) for i in range(45, 49)], 
    'chhattisgarh': ['49'],
    'andhra pradesh': ['50', '51', '52', '53'], 
    'telangana': ['50', '51', '52', '53'],
    'karnataka': [str(i) for i in range(56, 60)], 
    'tamil nadu': [str(i) for i in range(60, 65)],
    'puducherry': ['60', '53', '67'], 
    'kerala': ['67', '68', '69'], 
    'lakshadweep': ['68'],
    'west bengal': ['70', '71', '72', '73', '74'], 
    'sikkim': ['73'],
    'andaman and nicobar islands': ['74'], 
    'odisha': ['75', '76', '77'],
    'assam': ['78'], 
    'arunachal pradesh': ['78', '79'], 
    'manipur': ['79'], 
    'meghalaya': ['78', '79'], 
    'mizoram': ['79'], 
    'nagaland': ['79'], 
    'tripura': ['79'],
    'bihar': ['80', '81', '82', '83', '84', '85'], 
    'jharkhand': ['81', '82', '83']
}


In [6]:
district_maping = {
    #Andaman and Nicobar Islands
    'Andamans': 'South Andaman',
    'Nicobars': 'Nicobar',

    # Andhra Pradesh
    'Anantapur': 'Ananthapuramu',
    'Ananthapur': 'Ananthapuramu',
    'Y. S. R': 'YSR',  
    'Cuddapah': 'YSR', 
    'N. T. R': 'NTR',
    'Nellore': 'Sri Potti Sriramulu Nellore',
    'Kadiri Road': 'Sri Sathya Sai',  # Inferred: Kadiri is a major town in Sri Sathya Sai district
    'Spsr Nellore': 'Sri Potti Sriramulu Nellore',

    # Arunachal Pradesh
    'Shi-Yomi': 'Shi Yomi',  

    # Assam
    'South Salmara Mankachar': 'South Salmara-Mankachar',
    'Kamrup Metro': 'Kamrup Metropolitan',
    'Karimganj': 'Sribhumi',  # Renamed Nov 2024
    'Marigaon': 'Morigaon',  # Official spelling
    'Sibsagar': 'Sivasagar',  # Official spelling
    'North Cachar Hills': 'Dima Hasao',  # Renamed
    'Tamulpur District': 'Tamulpur',


    # Bihar
    'Purnia': 'Purnia',  # Standard Census spelling
    'Purnea': 'Purnia',  # Standardized to Purnia
    'Pashchim Champaran': 'West Champaran',  # Hindi -> English translation
    'Aurangabad(Bh)': 'Aurangabad',
    'Monghyr': 'Munger',  # Old spelling
    'Samstipur': 'Samastipur',  # Typo corrected
    'Sheikpura': 'Sheikhpura',  # Typo corrected
    'Purba Champaran': 'East Champaran',  # Hindi -> English translation
    'Bhabua': 'Kaimur (Bhabua)',  # HQ mapped to District
    'Near University Thana': 'Darbhanga',  # Location mapped to District
    'Purbi Champaran': 'East Champaran',   # Hindi -> English translation

    # Chhattisgarh
    'Baloda Bazar': 'Baloda Bazar-Bhatapara',
    'Balrampur': 'Balrampur-Ramanujganj',
    'Gariyaband': 'Gariaband',  # Official spelling
    'Dakshin Bastar Dantewada': 'Dantewada',  # Standardized to short name
    'Kawardha': 'Kabirdham',  # Official name changed in 2003
    'Uttar Bastar Kanker': 'Kanker',  # Standardized to short name
    'Kabeerdham': 'Kabirdham',
    'Manendragarhchirmiribharatpur': 'Manendragarh-Chirmiri-Bharatpur',
    'Khairagarh Chhuikhadan Gandai': 'Khairagarh-Chhuikhadan-Gandai',
    'Janjgir Champa': 'Janjgir-Champa',
    'Janjgir - Champa': 'Janjgir-Champa',
    'Manendragarh–Chirmiri–Bharatpur': 'Manendragarh-Chirmiri-Bharatpur',
    'Mohalla-Manpur-Ambagarh Chowki': 'Mohla-Manpur-Ambagarh Chowki',
    'Gaurella Pendra Marwahi': 'Gaurela-Pendra-Marwahi',

    # Delhi
    'North East': 'North East Delhi',
    'Najafgarh': 'South West Delhi',

    # Goa 

    'Bardez': 'North Goa', # sub-districts of North Goa
    'Tiswadi': 'North Goa',# sub-districts of North Goa
    'Bicholim': 'North Goa',# sub-districts of North Goa

    # Gujarat
    'Kachchh': 'Kachchh',  # Also known as Kutch
    'Arvalli': 'Aravalli',  # Official spelling
    'Surendra Nagar': 'Surendranagar',
    'Mahesana': 'Mehsana',  # Official government spelling
    'Ahmadabad': 'Ahmedabad',
    'Dohad': 'Dahod',  # Old spelling
    'The Dangs': 'Dang',
    'Banas Kantha': 'Banaskantha',
    'Panch Mahals': 'Panchmahal',
    'Sabar Kantha': 'Sabarkantha',

    # Haryana
    'Yamuna Nagar': 'Yamunanagar',
    'Mewat': 'Nuh',  # Renamed 2016
    'Gurgaon': 'Gurugram',  # Renamed 2016
    'Akhera': 'Nuh',  # Village/Tehsil mapped to District

    # Himachal Pradesh
    'Lahul And Spiti': 'Lahaul and Spiti',  # Spelling correction

    # Jammu and Kashmir
    'Punch': 'Poonch',
    'Badgam': 'Budgam',  # Official spelling
    'Baramula': 'Baramulla',
    'Shupiyan': 'Shopian',
    'Rajauri': 'Rajouri',
    'Bandipur': 'Bandipora',
    '?': 'Jammu',  # searched through pincode

    # Jharkhand
    'Purbi Singhbhum': 'East Singhbhum',  # Hindi -> English translation
    'Pashchimi Singhbhum': 'West Singhbhum',  # Hindi -> English translation
    'Pakaur': 'Pakur',  # Official administration spelling
    'Sahebganj': 'Sahibganj',
    'Hazaribag': 'Hazaribagh',
    'Palamau': 'Palamu',  # Old spelling
    'East Singhbum': 'East Singhbhum',  # Typo correction

    # Karnataka
    'Davangere': 'Davanagere',  # Official spelling includes the middle 'a'
    'Tumkur': 'Tumakuru',
    'Belgaum': 'Belagavi',
    'Chamrajanagar': 'Chamarajanagar',
    'Bengaluru': 'Bengaluru Urban',
    'Shimoga': 'Shivamogga',
    'Bellary': 'Ballari',
    'Mysore': 'Mysuru',
    'Hasan': 'Hassan',
    'Bangalore': 'Bengaluru Urban',
    'Bijapur': 'Vijayapura',
    'Ramanagar': 'Ramanagara',  # Official spelling ends with 'a'
    'Chickmagalur': 'Chikkamagaluru',
    'Gulbarga': 'Kalaburagi',
    'Bangalore Rural': 'Bengaluru Rural',
    'Chamrajnagar': 'Chamarajanagar',
    'Chikmagalur': 'Chikkamagaluru',
    'Bengaluru South': 'Bengaluru Urban',  # This is a Taluk/Zone within Urban
    'Bijapur(Kar)': 'Vijayapura',
    '5Th Cross': 'Bengaluru Urban',  # Inferred: Common street address format in Bangalore

    # Kerala,
    'Kasargod': 'Kasaragod',

    # Union Territory of Ladakh
    'Leh (Ladakh)': 'Leh', 

    # Madhya Pradesh
    'West Nimar': 'Khargone',  # Standardized to HQ name
    'East Nimar': 'Khandwa',  # Standardized to HQ name
    'Ashok Nagar': 'Ashoknagar',  # Standardized to one word
    'Hoshangabad': 'Narmadapuram',  # Old name mapped to new official name
    'Narsimhapur': 'Narsinghpur',

    # Maharashtra
    'Ahmadnagar': 'Ahilyanagar',  # Renamed Oct 2024
    'Gondiya': 'Gondia',  # Official spelling used by administration
    'Raigarh': 'Raigad',  # 'Raigarh' is in Chhattisgarh; MH district is Raigad
    'Aurangabad': 'Chhatrapati Sambhajinagar',  # Renamed
    'Mumbai': 'Mumbai City',  # Ambiguous, mapped to main City district
    'Buldana': 'Buldhana',  # Official spelling
    'Osmanabad': 'Dharashiv',  # Renamed
    'Bid': 'Beed',  # Old spelling
    'Chatrapati Sambhaji Nagar': 'Chhatrapati Sambhajinagar',
    'Ahmed Nagar': 'Ahilyanagar',  # Renamed
    'Mumbai( Sub Urban )': 'Mumbai Suburban',
    'Raigarh(Mh)': 'Raigad',
    'Dist : Thane': 'Thane',
    'Near Uday Nagar Nit Garden': 'Nagpur',  # Location mapped to District (NIT = Nagpur Improvement Trust)
    'Ahmednagar': 'Ahilyanagar',  # Renamed

    # Meghalaya
    'Jaintia Hills': 'West Jaintia Hills',  # Split in 2012; HQ Jowai is in West
    'Kamrup': 'Ri Bhoi',  # Kamrup is in Assam; likely border confusion with Ri Bhoi

    # Mizoram  
    'Mammit': 'Mamit',  # Typo correction
    'Saiha': 'Siaha',  # Official spelling changed to Siaha
   
    # Nagaland
    'Chumukedima': 'Chümoukedima',  # Official spelling includes 'ou' and diacritic

    # Odisha
    'Sundergarh': 'Sundargarh',
    'Khorda': 'Khordha',
    'Baleswar': 'Balasore',  # Official English name
    'Jajapur': 'Jajpur',
    'Sonapur': 'Subarnapur',  # Official name
    'Baudh': 'Boudh',
    'Baleshwar': 'Balasore',
    'Kendujhar': 'Keonjhar',  # Official English name
    'Anugul': 'Angul',
    'Jagatsinghapur': 'Jagatsinghpur',
    'Debagarh': 'Deogarh',  # Official English name
    'Anugal': 'Angul',
    'Balianta': 'Khordha',  # Block/Tehsil mapped to District
    'Bhadrak(R)': 'Bhadrak',


    # Union Territory of Puducherry
    'Pondicherry': 'Puducherry',  # Renamed 2006


    # Punjab
    'Sas Nagar (Mohali)': 'Sahibzada Ajit Singh Nagar',
    'Firozpur': 'Ferozepur',
    'S.A.S Nagar(Mohali)': 'Sahibzada Ajit Singh Nagar',
    'Muktsar': 'Sri Muktsar Sahib',  # Renamed
    'Nawanshahr': 'Shaheed Bhagat Singh Nagar',  # Renamed
    'S.A.S Nagar': 'Sahibzada Ajit Singh Nagar',

    'Mohali': 'Sahibzada Ajit Singh Nagar'  ,

    # Rajasthan

    'Jhunjhunun': 'Jhunjhunu',  # Standardized spelling
    'Jalor': 'Jalore',
    'Ganganagar': 'Sri Ganganagar',  # Official name includes 'Sri'
    'Chittaurgarh': 'Chittorgarh',  # Phonetic correction
    'Dhaulpur': 'Dholpur',  # Phonetic correction
    #        'Near Meera Hospital': 'Unknown'  # Address data, not a district (Likely Ajmer, Sikar, or Jaipur)
    
    # Sikkim
    'East': 'Gangtok',
    'East Sikkim': 'Gangtok',
    'South': 'Namchi',
    'South Sikkim': 'Namchi',
    'North Sikkim': 'Mangan',
    'West': 'Gyalshing',
    'North': 'Mangan',
    'West Sikkim': 'Gyalshing',


    # Tamil Nadu,
    'Thiruvarur': 'Tiruvarur',  # Official spelling
    'Thoothukkudi': 'Thoothukudi',  # Official spelling
    'Kanyakumari': 'Kanniyakumari',  # Spelling correction
    'Thiruvallur': 'Tiruvallur',  # Spelling correction
    'Tirupathur': 'Tirupattur',  # Spelling correction
    'Tuticorin': 'Thoothukudi',  # Renamed
    'Kanchipuram': 'Kancheepuram',  # Spelling correction
    'Near Dhyana Ashram': 'Chennai',  # Location mapped to District (Mylapore, Chennai)


    # Telangana
    'K.V. Rangareddy': 'Rangareddy',
    'Warangal Rural': 'Warangal',  # Renamed 2021
    'Yadadri.': 'Yadadri Bhuvanagiri',  # Official Full Name
    'Komaram Bheem': 'Kumuram Bheem Asifabad',  # Official Full Name
    'Jagitial': 'Jagtial',  # Official Spelling
    'Warangal Urban': 'Hanumakonda',  # Renamed 2021
    'Medchal?Malkajgiri': 'Medchal-Malkajgiri',  # Encoding fix
    'Medchal−Malkajgiri': 'Medchal-Malkajgiri',  # Encoding fix
    'Warangal (Urban)': 'Hanumakonda',  # Renamed 2021
    'Idpl Colony': 'Medchal-Malkajgiri',  # Locality in Balanagar Mandal
    'Medchalâ\x88\x92Malkajgiri': 'Medchal-Malkajgiri',  # Encoding fix
    'Medchal Malkajgiri': 'Medchal-Malkajgiri',
    'Ranga Reddy': 'Rangareddy',

    'Mahbubnagar': 'Mahabubnagar',  
    'Mahabub Nagar': 'Mahabubnagar',  
    'Karim Nagar': 'Karimnagar',  
    'K.V. Rangareddy': 'Ranga Reddy',  
    'K.V.Rangareddy': 'Ranga Reddy',  


    # Uttar Pradesh
    'Bara Banki': 'Barabanki',
    'Allahabad': 'Prayagraj',  # Renamed 2018
    'Faizabad': 'Ayodhya',  # Renamed 2018
    'Sant Ravidas Nagar': 'Bhadohi',  # Renamed 2014
    'Kheri': 'Lakhimpur Kheri',  # Official name often used with HQ
    'Rae Bareli': 'Raebareli',
    'Noida': 'Gautam Buddha Nagar',  
    'Shrawasti': 'Shravasti',  # Standard spelling
    'Sant Ravidas Nagar Bhadohi': 'Bhadohi',
    'Jyotiba Phule Nagar': 'Amroha',  # Renamed 2012
    'Bagpat': 'Baghpat',
    'Mahrajganj': 'Maharajganj',
    'Siddharth Nagar': 'Siddharthnagar',
    'Kushi Nagar': 'Kushinagar',

    # Uttarakhand
    'Garhwal': 'Pauri Garhwal',  # Historical naming convention
    'Hardwar': 'Haridwar'  # Old spelling

}

# Andra pradesh to  Telangana districts list
telangana_districts = [
    'Warangal', 'Adilabad', 'Hyderabad', 'Nizamabad', 'Khammam', 
    'Karimnagar', 'Medak', 'Mahabubnagar', 'Nalgonda', 'Ranga Reddy'
]

# Chandigarh to panjab change 
punjab_districts = ['Rupnagar', 'Sahibzada Ajit Singh Nagar']

# jammu and kashmir leh district to ladakh change
ladakh_districts = ['Leh', 'Kargil']

# puducherry to tamil nadu  change
tn_districts = ['Viluppuram', 'Cuddalore']

In [7]:
def clean_data(file_path, file_name):
    data = pd.read_csv(file_path + file_name)

    data['date'] = pd.to_datetime(data['date'], format='%d-%m-%Y')
    data.rename(columns=rename_map, inplace=True)

    data['state'] = data['state'].astype(str).str.lower().str.strip().str.replace('&', 'and', regex=False)
    data = data[data['state'] != '100000']
    data['state'] = data['state'].replace(state_map)

    # Removing * and converting & to and
    data['district'] = data['district'].astype(str).str.lower().str.replace('&', 'and', regex=False).str.replace('*', '', regex=False).str.strip().str.title()
    data['district'] = data['district'].replace(district_maping)

    # 1. Andhra Pradesh -> Telangana
    data.loc[data['district'].isin(telangana_districts), 'state'] = 'telangana'

    # 2. Chandigarh -> Punjab
    data.loc[data['district'].isin(punjab_districts), 'state'] = 'punjab'

    # 3. J&K -> Ladakh
    data.loc[data['district'].isin(ladakh_districts), 'state'] = 'ladakh'

    # 4. Puducherry -> Tamil Nadu
    data.loc[data['district'].isin(tn_districts), 'state'] = 'tamil nadu'

    data['district'] = data['district'].astype(str).str.lower()
    data['state'] = data['state'].astype(str).str.lower()

    states = data['state'].astype(str).str.lower().str.strip()
    prefixes = data['pincode'].astype(str).str.strip().str[:2]
    
    valid_mask = []
    
    for state, prefix in zip(states, prefixes):
        if state == 'chandigarh' and prefix == '14':
            valid_mask.append(True) 
        elif state in state_prefixes and prefix not in state_prefixes[state]:
            valid_mask.append(False) 
        else:
            valid_mask.append(True) 
            
    data = data[valid_mask]

    return data

In [ ]:
for i in [0,1,2]:
    data_list = []
    if i==0:
        file = api_data_aadhar_biometric
        name = "api_data_aadhar_biometric.csv"
    elif i==1:
        file = api_data_aadhar_demographic
        name = "api_data_aadhar_demographic.csv"
    elif i== 2:
        file = api_data_aadhar_enrolment
        name = "api_data_aadhar_enrolment.csv"

    for file_name in file:
        data = clean_data(aadhar_dataset[i],file_name)
        data_list.append(data)

    if data_list:
        file = pd.concat(data_list, ignore_index=True)
        display(f" Total rows: {len(file)}")
    
    df = pd.DataFrame(file)
    df.to_csv(save_path + name)

' Total rows: 1860617'

' Total rows: 2071083'

' Total rows: 1005857'

In [10]:
data_list = pd.DataFrame()

file_groups = {
    0: api_data_aadhar_biometric,
    1: api_data_aadhar_demographic,
    2: api_data_aadhar_enrolment
}

for i, file_list in file_groups.items():
    for file_name in file_list:

        data = clean_data(aadhar_dataset[i],file_name)
        df = pd.DataFrame(data[['district','state' , 'pincode']]).drop_duplicates()
        data_list = pd.concat([data_list, df], ignore_index=True)

data_list = data_list.drop_duplicates()

data_list.to_csv('cleaned dataset/geo/data_list.csv', index=False)